In [ ]:
######################### Batch wise processing for zauba first result from dropdown using selenium ####################################################
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


# === List of realistic user agents (desktop + mobile mix) ===
USER_AGENTS = [
    # Chrome (Windows)
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    # # Chrome (Mac)
    # "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    # "(KHTML, like Gecko) Chrome/118.0.5993.90 Safari/537.36",
    # Edge
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/118.0.5993.90 Safari/537.36 Edg/118.0.2088.61",
    # Firefox
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0",
    # # Mobile (Android Chrome)
    # "Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 "
    # "(KHTML, like Gecko) Chrome/119.0.6045.134 Mobile Safari/537.36",
    # # Mobile (iPhone)
    # "Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) AppleWebKit/605.1.15 "
    # "(KHTML, like Gecko) Version/17.0 Mobile/15E148 Safari/604.1",
]


def start_driver():
    """Initialize and return a new Chrome WebDriver instance with random user-agent."""
    ua = random.choice(USER_AGENTS)
    print(f"🧠 Using User-Agent: {ua}\n")

    service = Service(ChromeDriverManager().install())
    options = Options()
    options.add_argument("--disable-notifications")
    options.add_argument("--start-maximized")
    options.add_argument(f"user-agent={ua}")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    driver = webdriver.Chrome(service=service, options=options)
    return driver


# === Load Input Data ===
# input_list = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\Book2.xlsx")['c_name'].to_list()[:]
false_df = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\Tofler_data_2.xlsx")
input_list = false_df[false_df['is_match'] == False]['input'].to_list()
# === Output File ===
output_path = "Zauba_data_2.xlsx"
all_data = []

# === Resume if partial file exists ===
if os.path.exists(output_path):
    existing_df = pd.read_excel(output_path)
    processed_names = set(existing_df['input'])
    input_list = [name for name in input_list if name not in processed_names]
    all_data = existing_df.to_dict('records')
    print(f"🟡 Resuming — already processed {len(processed_names)} companies.")
else:
    print("🟢 Starting fresh extraction.")

batch_size = 50
driver = start_driver()

for idx, input_name in enumerate(input_list, start=1):
    try:
        driver.get("https://www.zaubacorp.com/")
        time.sleep(2)

        # Clean company name for first attempt
        clean_name = input_name.strip().rstrip(',')
        driver.find_element(By.ID, "searchid").send_keys(clean_name)
        time.sleep(2)
        driver.find_element(By.ID, "result").find_element(By.TAG_NAME, "div").click()
        time.sleep(2)
        try:
            corrected_name = driver.find_element(By.ID, 'title').text
        except Exception as e:
            print(e)
            corrected_name = driver.current_url.split("/")[-1].rsplit("-", 1)[0].replace("-", " ")
        cin = driver.current_url.split("/")[-1].rsplit("-", 1)[1]

        print({"input": input_name, "cin": cin, "corrected_name": corrected_name})
        all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin})

    except Exception:
        print("❌ Failed — retrying with simplified name:", input_name)
        try:
            driver.get("https://www.zaubacorp.com/")
            time.sleep(2)

            simplified = (
                input_name.lower()
                .replace("limited.", "")
                .replace("private.", "")
                .replace("limited", "")
                .replace("private", "")
                .replace("pvt.", "")
                .replace("ltd.", "")
                .replace("pvt", "")
                .replace("ltd", "")
                .replace("llp.", "")
                .replace("llp", "")
                .replace("messers", "")
                .replace("messars", "")
                .replace("m/s", "")
                .strip()
                .rstrip(',')
            )

            driver.find_element(By.ID, "searchid").send_keys(simplified)
            time.sleep(2)
            driver.find_element(By.ID, "result").find_element(By.TAG_NAME, "div").click()
            time.sleep(2)
            try:
                corrected_name = driver.find_element(By.ID, 'title').text
            except Exception as e:
                print(e)
                corrected_name = driver.current_url.split("/")[-1].rsplit("-", 1)[0].replace("-", " ")
            cin = driver.current_url.split("/")[-1].rsplit("-", 1)[1]

            print({"input": input_name, "cin": cin, "corrected_name": corrected_name})
            all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin})
            print()

        except Exception as e:
            print("🚫 Not found:", input_name)
            all_data.append({"input": input_name, "corrected_name": None, "cin": None})

    # === Restart driver + Save progress after every 100 ===
    if idx % batch_size == 0:
        print(f"\n💾 Saving progress and restarting browser after {idx} companies...\n")
        pd.DataFrame(all_data).to_excel(output_path, index=False)
        print(f"✅ Progress saved to {output_path}")

        driver.quit()
        time.sleep(25)
        driver = start_driver()

# === Final save ===
driver.quit()
if all_data:
    pd.DataFrame(all_data).to_excel(output_path, index=False)
    print(f"🎉 All done! Final file saved to {output_path}")


🟡 Resuming — already processed 571 companies.
🧠 Using User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0



KeyboardInterrupt: 

In [ ]:
######################### Batch-wise Headless Zauba Scraper with multiple results from dropdown using selenium####################################################
import os
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import re


# === List of realistic user agents (desktop + mobile mix) ===
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/118.0.5993.90 Safari/537.36 Edg/118.0.2088.61",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0",
]


def human_delay(a=2, b=5):
    """Random sleep between a and b seconds."""
    time.sleep(random.uniform(a, b))


def start_driver(headless=True):
    """Initialize and return a new Chrome WebDriver instance with random user-agent."""
    ua = random.choice(USER_AGENTS)
    print(f"🧠 Using User-Agent: {ua}\n")

    service = Service(ChromeDriverManager().install())
    options = Options()
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(f"user-agent={ua}")

    # if headless:
    #     options.add_argument("--headless=new")
    #     options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(service=service, options=options)
    return driver


# === Load Input Data ===
# false_df = pd.read_excel(r"D:\Nexensus_Projects\pdfFoms\Tofler_data_2.xlsx")

# input_df = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\Book4_data.xlsx")
# input_list = input_df[input_df['cin'].isna()]['input'].tolist()

input_list = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\input_folder\NCLT_17-11-2025.xlsx")['c_name'].to_list()[1175:]

# === Output File ===
output_path = "NCLT_17-11-2025_1175-.xlsx"
all_data = []

# === Resume if partial file exists ===
if os.path.exists(output_path):
    existing_df = pd.read_excel(output_path)
    processed_names = set(existing_df['input'])
    input_list = [name for name in input_list if name not in processed_names]
    all_data = existing_df.to_dict('records')
    print(f"🟡 Resuming — already processed {len(processed_names)} companies.")
else:
    print("🟢 Starting fresh extraction.")

batch_size = 25
driver = start_driver(headless=True)


# === Helper to simplify names ===
def simplify_name(name):
    return (
        name.lower()
        .replace("limited.", "")
        .replace("private.", "")
        .replace("limited", "")
        .replace("private", "")
        .replace("pvt.", "")
        .replace("ltd.", "")
        .replace("pvt", "")
        .replace("ltd", "")
        .replace("llp.", "")
        .replace("llp", "")
        .replace("messers", "")
        .replace("messars", "")
        .replace("m/s", "")
        .replace("()", "")
        .strip()
        .rstrip(',')
    )

def clean_excel_string(s):
    if not isinstance(s, str):
        return s
    # Remove illegal characters for Excel
    return re.sub(r'[^\x09\x0A\x0D\x20-\uD7FF\uE000-\uFFFD]', '', s)

# === Core logic to fetch multiple results ===
def process_company(driver, input_name):
    """Search company, click all results, and capture data."""
    results_data = []
    clean_name = input_name.strip().rstrip(',')
    search_attempts = [clean_name, simplify_name(input_name)]

    for attempt_no, search_term in enumerate(search_attempts, start=1):
        try:
            driver.get("https://www.zaubacorp.com/")
            human_delay()

            search_box = driver.find_element(By.ID, "searchid")
            search_box.clear()
            search_box.send_keys(search_term)
            human_delay()

            result_div = driver.find_element(By.ID, "result")
            result_items = result_div.find_elements(By.TAG_NAME, "div")

            if not result_items:
                print(f"⚠️ No results found for attempt {attempt_no}: {search_term}")
                continue

            print(f"🔍 Found {len(result_items)} results for: {search_term}")

            for i in range(len(result_items)):
                try:
                    # Reopen and search again each time to avoid stale elements
                    driver.get("https://www.zaubacorp.com/")
                    human_delay()
                    search_box = driver.find_element(By.ID, "searchid")
                    search_box.clear()
                    search_box.send_keys(search_term)
                    human_delay()

                    fresh_results = driver.find_element(By.ID, "result").find_elements(By.TAG_NAME, "div")

                    if i >= len(fresh_results):
                        continue

                    print(f"➡️ Clicking result {i + 1}/{len(fresh_results)} for {input_name}")
                    fresh_results[i].click()
                    human_delay()

                    current_url = driver.current_url
                    try:
                        corrected_name = driver.find_element(By.ID, "title").text
                    except Exception:
                        corrected_name = current_url.split("/")[-1].rsplit("-", 1)[0].replace("-", " ")

                    cin = None
                    if "-" in current_url.split("/")[-1]:
                        if "llp" in corrected_name.lower():
                            cin = current_url.split("/")[-1].rsplit("-", 1)[0].rsplit("-", 1)[1] + " " + current_url.split("/")[-1].rsplit("-", 1)[1]
                        else:
                            cin = current_url.split("/")[-1].rsplit("-", 1)[1]

                    print(f"✅ {corrected_name} — {cin}")
                    results_data.append({
                        "input": input_name,
                        "search_term": search_term,
                        "corrected_name": corrected_name,
                        "cin": cin,
                        "url": current_url
                    })

                    # === Stop further dropdown clicks if corrected name matches cleaned input ===
                    cleaned_input = simplify_name(input_name)
                    cleaned_corrected = simplify_name(corrected_name)
                    if cleaned_input == cleaned_corrected:
                        print(f"🛑 Match found for '{input_name}' — stopping further dropdown clicks.")
                        break

                except Exception as e:
                    print(f"❌ Error on {input_name} result {i+1}: {e}")
                    continue

            if results_data:
                break  # stop after successful search
        except Exception as e:
            print(f"🚫 Search failed for attempt {attempt_no}: {search_term} — {e}")
            driver.quit()
            time.sleep(random.randint(10, 20))
            driver = start_driver(headless=True)
            driver.get("https://www.zaubacorp.com/")
            human_delay()
            continue

    if not results_data:
        results_data.append({
            "input": input_name,
            "search_term": None,
            "corrected_name": None,
            "cin": None,
            "url": None
        })

    return results_data


# === Main Loop ===
for idx, input_name in enumerate(input_list, start=1):
    input_name = input_name.strip()
    company_results = process_company(driver, input_name)
    all_data.extend(company_results)

    # === Restart + save every batch ===
    if idx % batch_size == 0:
        print(f"\n💾 Saving progress and restarting browser after {idx} companies...\n")
        output_df = pd.DataFrame(all_data).applymap(clean_excel_string)
        output_df.to_excel(output_path, index=False)
        # pd.DataFrame(all_data).to_excel(output_path, index=False)
        print(f"✅ Progress saved to {output_path}")
        driver.quit()
        time.sleep(random.randint(10, 20))
        driver = start_driver(headless=True)

# === Final Save ===
driver.quit()
if all_data:
    output_df = pd.DataFrame(all_data).applymap(clean_excel_string)
    output_df.to_excel(output_path, index=False)
    # pd.DataFrame(all_data).to_excel(output_path, index=False)
    print(f"🎉 All done! Final file saved to {output_path}")


🟢 Starting fresh extraction.
🧠 Using User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36

⚠️ No results found for attempt 1: Shree Raghuvanshi Fibres Pvt Ltd
🚫 Search failed for attempt 2: shree raghuvanshi fibres — Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=142.0.7444.176)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xb94103
	0xb94144
	0x99e71d
	0x98d910
	0x9ac714
	0xa13475
	0xa2e749
	0xa0c706
	0x9dda30
	0x9ded54
	0xe057b4
	0xe0098a
	0xbbc392
	0xbac4c8
	0xbb324d
	0xb9c478
	0xb9c63c
	0xb867ca
	0x757ffcc9
	0x779d82ae
	0x779d827e

🧠 Using User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0



KeyboardInterrupt: 

In [ ]:
############################### Zauba fetching fist 10 result using requests ################################
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd


input_list = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\input_folder\Highcourt.xlsx")['c_name'].drop_duplicates().to_list()[:100]

# input_name = "Bengal Paper Industries Ltd".replace(" ", "-")
all_data = []
for input_name in input_list:
    # print("current input :", input_name)
    url = "https://www.zaubacorp.com/companysearchresults/{}".format(input_name.replace(" ", "-"))

    resp = requests.get(
        url,
        impersonate="chrome",
        cookies={  # put your cookies here
            # example:
            # "cf_clearance": "...",
            # "ZCSESSID": "...",
        }
    )

    # print(resp.status_code)

    soup = BeautifulSoup(resp.text, 'html.parser')
    try:
        result_table = soup.find("table", {"id": "results"})
        if len(result_table.find_all('tr')) >1:
            for row in result_table.find_all('tr')[1:11]:
                # print(row)
                c_page_link = row.find_all('td')[0].find('a')["href"]
                print(c_page_link)
                cin = row.find_all('td')[0].text.strip()
                corrected_name = row.find_all('td')[1].text.strip()
                address = row.find_all('td')[2].text.strip()

                company_page_response = requests.get(
                    c_page_link,
                    impersonate="chrome",
                    cookies={  # put your cookies here
                        # example:
                        # "cf_clearance": "...",
                        # "ZCSESSID": "...",
                    }
                )
                company_page_soup = BeautifulSoup(company_page_response.text, 'html.parser')
                all_rc_divs = company_page_soup.find_all("div", class_='rc')
                previous_names = []
                for rc_div in all_rc_divs:
                    if rc_div.find('h3').text.strip() == "Previous Names":
                        for row in rc_div.find("table").find_all('tr'):
                            previous_names.append(row.find('td').text.strip())

                print(previous_names)
                # print("cin :", cin)
                # print("corrected_name :", corrected_name)
                all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin, "link" : c_page_link, "previous_names" : previous_names})
        else:
            # all_data.append({"input": input_name, "corrected_name": None, "cin": None})
            simplified = (
                input_name.lower()
                .replace("(p) limited.", "")
                .replace("(p) limited", "")  
                .replace("p limited.", "")  
                .replace("p Limited", "")  
                .replace("(p) ltd.", "")    
                .replace("(p).ltd.", "")
                .replace("p.Ltd.", "")  
                .replace("p Ltd.", "")  
                .replace("pLtd.", "")  
                .replace("(p) Ltd", "")
                .replace("(p)Ltd", "")
                .replace("pLtd", "")  
                .replace("p Ltd", "")          
                .replace("limited.", "")
                .replace("private.", "")
                .replace("limited", "")
                .replace("private", "")
                .replace("pvt.", "")
                .replace("ltd.", "")
                .replace("pvt", "")
                .replace("ltd", "")
                .replace("llp.", "")
                .replace("llp", "")
                .replace("messers", "")
                .replace("messars", "")
                .replace("m/s", "")
                .replace("()", "")
                # .replace("A.O.,", "")
                .strip()
                .rstrip(',')
                .lstrip(',')
                .lstrip(".")
            )
            # print("retrying with simplified name :", simplified)
            url = "https://www.zaubacorp.com/companysearchresults/{}".format(simplified.replace(" ", "-"))

            resp = requests.get(
                url,
                impersonate="chrome",
                cookies={  # put your cookies here
                    # example:
                    # "cf_clearance": "...",
                    # "ZCSESSID": "...",
                }
            )

            # print(resp.status_code)

            soup = BeautifulSoup(resp.text, 'html.parser')
            try:
                result_table = soup.find("table", {"id": "results"})
                if len(result_table.find_all('tr')) >1:
                    for row in result_table.find_all('tr')[1:11]:
                        # print(row)
                        c_page_link = row.find_all('td')[0].find('a')["href"]
                        print(c_page_link)
                        cin = row.find_all('td')[0].text.strip()
                        corrected_name = row.find_all('td')[1].text.strip()
                        address = row.find_all('td')[2].text.strip()
                        # print("cin :", cin)
                        # print("corrected_name :", corrected_name)

                        company_page_response = requests.get(
                            c_page_link,
                            impersonate="chrome",
                            cookies={  # put your cookies here
                                # example:
                                # "cf_clearance": "...",
                                # "ZCSESSID": "...",
                            }
                        )
                        company_page_soup = BeautifulSoup(company_page_response.text, 'html.parser')
                        all_rc_divs = company_page_soup.find_all("div", class_='rc')
                        previous_names = []
                        for rc_div in all_rc_divs:
                            if rc_div.find('h3').text.strip() == "Previous Names":
                                for row in rc_div.find("table").find_all('tr'):
                                    previous_names.append(row.find('td').text.strip())

                        print(previous_names)

                        all_data.append({"input": input_name, "corrected_name": corrected_name, "cin": cin, "link" : c_page_link, "previous_names" : previous_names})
                else:
                    all_data.append({"input": input_name, "corrected_name": None, "cin": None, "link" : None, "previous_names" : None})
            except Exception as e:
                print(e)
                all_data.append({"input": input_name, "corrected_name": None, "cin": None, "link" : None, "previous_names" : None})

    except Exception as e:
        print(e)
        all_data.append({"input": input_name, "corrected_name": None, "cin": None, "link" : None, "previous_names" : None})


from fuzzywuzzy import fuzz
import pandas as pd
output_path = "HC_Zauba_data.xlsx"


if all_data:
    data_df = pd.DataFrame(all_data)
    # data_df["match_score"] = data_df.apply(
    #     lambda row: fuzz.partial_ratio(row["input"], row["corrected_name"]),
    #     axis=1
    # )
    data_df.to_excel(output_path, index=False)
    # pd.DataFrame(all_data).to_excel(output_path, index=False)
    print(f"🎉 All done! Final file saved to {output_path}")



https://www.zaubacorp.com/THE-FEDERAL-BANK-LTD-L65191KL1931PLC000368
['THE FEDERAL BANK LTD']
https://www.zaubacorp.com/TRAVANCORE-GENERAL-BANK-LTD-U65191KL1928PLC001157
['TRAVANCORE GENERAL BANK LTD']
https://www.zaubacorp.com/BANKA-GENERAL-FINANCE-AND-INVESTMENT-LTD-U65921PB1988PLC008902
['BANKA GENERAL FINANCE AND INVESTMENT LTD']
https://www.zaubacorp.com/ABAAN-BUILDERS-AND-LAND-DEVELOPERS-PRIVATE-LIMITED-U45203MP2014PTC032898
['ABAAN BUILDERS AND LAND DEVELOPERS PRIVATE LIMITED', 'ABAAN BUILDERS AND LAND DEVELOPERSPRIVATE LIMITED']
https://www.zaubacorp.com/AZAAN-BUILDERS-PRIVATE-LIMITED-U45202UP1997PTC022169
['AZAAN BUILDERS PRIVATE LIMITED']
https://www.zaubacorp.com/ABASAN-BUILDERS-PRIVATE-LIMITED-U99999DL1992PTC051009
['ABASAN BUILDERS PRIVATE LIMITED']
https://www.zaubacorp.com/ABAAM-BUILDERS-PRIVATE-LIMITED-U45400MH2011PTC221091
['ABAAM BUILDERS PRIVATE LIMITED']
https://www.zaubacorp.com/AHAAN-BUILDERS-PRIVATE-LIMITED-U45201DL2005PTC138218
['AHAAN BUILDERS PRIVATE LIMITED']

5af9ecd6dde388230086089d06c79afd
1d037a85f0c187d5587a26e50a41ae15


In [1]:
############ Block to fetch previous names ##########################################
from curl_cffi import requests
from bs4 import BeautifulSoup
import pandas as pd

df = pd.read_excel(r"D:\Nexensus_Projects\ToflerAndZuba\output_folder\08-12-2025_data.xlsx")[:]

previous_names_list = []
old_name_match_list = []   # <-- store match result for each row

for row in df.itertuples(index=False):
    cin = row.cin
    input_name = row.input

    if isinstance(cin, float) or cin is None:
        previous_names_list.append([])
        old_name_match_list.append(False)   # no CIN → no match
        continue

    print(row.corrected_name)

    url = f"https://www.zaubacorp.com/companysearchresults/{cin}"
    company_page_response = requests.get(
        url,
        impersonate="chrome",
        cookies={}
    )

    soup = BeautifulSoup(company_page_response.text, 'html.parser')
    all_rc_divs = soup.find_all("div", class_='rc')

    previous_names = []
    for rc_div in all_rc_divs:
        h3 = rc_div.find('h3')
        if h3 and h3.text.strip() == "Previous Names":
            table = rc_div.find("table")
            if table:
                for tr in table.find_all("tr"):
                    td = tr.find('td')
                    if td:
                        previous_names.append(td.text.strip())

    previous_names_list.append(previous_names)

    # Normalize strings for safe matching
    normalized_input = input_name.lower().strip()
    normalized_prev = [name.lower().strip() for name in previous_names]

    match = normalized_input in normalized_prev
    old_name_match_list.append(match)

    if match:
        print("MATCH:", normalized_input)
        print("IN:", normalized_prev)

# Add new columns
df["previous_names"] = previous_names_list
df["old_name_match"] = old_name_match_list

df.to_excel("updated_with_previous_names.xlsx", index=False)

print("New column added successfully!")


LULU INDIA SHOPPING MALL PRIVATE LIMITED
GANTON PROJECTS PRIVATE LIMITED
MATCH: ganton projects private limited
IN: ['ganton aviation private limited', 'ganton projects private limited']
SAMSON AND SONS BUILDERS AND DEVELOPERS PRIVATE LIMITED
MATCH: samson and sons builders and developers private limited
IN: ['samson and sons builders and developers private li mited', 'samson and sons builders and developers private limited', 'samson and sons builders and developersprivate limited']
SAMSON AND SONS BUILDERS AND DEVELOPERS PRIVATE LIMITED (old name)
MATCH: samson and sons builders and developers private limited
IN: ['samson and sons builders and developers private li mited', 'samson and sons builders and developers private limited', 'samson and sons builders and developersprivate limited']
ABG SHIPYARD LTD
PAI BROTHERS ENGINEERS PRIVATE LIMITED
MATCH: pai brothers engineers private limited
IN: ['pai brothers engineers private limited']
SHIVAM SIGN-TECH PRIVATE LIMITED.
SHIVAM SIGN-TECH 

Exception ignored from cffi callback <function buffer_callback at 0x000001ECA5003240>:
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\curl_cffi\curl.py", line 100, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 


RequestException: Failed to perform, curl: (23) Failure writing output to destination, passed 13 returned 0. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.